# EdgeInspect-AI — DeepPCB training (separate experiment)

This notebook trains a new PCB model. It does not change the NEU-DET steel checkpoint. Attach two Kaggle inputs before running: (1) a dataset containing this project folder with `scripts/prepare_deeppcb.py`, `training/train_detector.py`, and `yolov8n.pt`; (2) the official DeepPCB source folder containing `trainval.txt`, `test.txt`, and its group folders. Review DeepPCB research-use terms before upload or redistribution.

Turn on a Kaggle GPU accelerator for training. Training output and test metrics are saved as a downloadable ZIP. Never report expected numbers as measured results.

In [ ]:
!pip install -q ultralytics==8.4.163 pyyaml pillow
import json, shutil, subprocess, sys, zipfile
from pathlib import Path
import torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before training.')

In [ ]:
input_root = Path('/kaggle/input')
project_scripts = list(input_root.rglob('prepare_deeppcb.py'))
project_scripts = [p for p in project_scripts if p.parent.name == 'scripts' and (p.parents[1] / 'training' / 'train_detector.py').is_file()]
if len(project_scripts) != 1:
    raise FileNotFoundError(f'Expected exactly one attached project copy with prepare_deeppcb.py; found {len(project_scripts)}. Attach the project dataset and inspect /kaggle/input.')
project_source = project_scripts[0].parents[1]
pcb_lists = [p for p in input_root.rglob('trainval.txt') if (p.parent / 'test.txt').is_file()]
if len(pcb_lists) != 1:
    raise FileNotFoundError(f'Expected one DeepPCB source folder containing trainval.txt and test.txt; found {len(pcb_lists)}. Attach the official source as a Kaggle input.')
pcb_source = pcb_lists[0].parent
work = Path('/kaggle/working/edgeinspect')
if work.exists(): shutil.rmtree(work)
shutil.copytree(project_source, work, ignore=shutil.ignore_patterns('.git', '.venv', 'dataset', 'runs', 'logs', '__pycache__'))
print('Project copy:', project_source)
print('DeepPCB source:', pcb_source)
print('Working directory:', work)

In [ ]:
processed = Path('/kaggle/working/deep_pcb')
subprocess.run([sys.executable, str(work / 'scripts' / 'prepare_deeppcb.py'), '--source', str(pcb_source), '--output', str(processed), '--seed', '42', '--val-fraction', '0.10'], check=True)
summary = json.loads((processed / 'summary.json').read_text())
print(json.dumps(summary['splits'], indent=2))
print('Data YAML:', processed / 'deep_pcb.yaml')

In [ ]:
# New experiment name: does not overwrite any steel experiment.
run_name = 'PCB-EXP-001'
subprocess.run([
    sys.executable, str(work / 'training' / 'train_detector.py'),
    '--data', str(processed / 'deep_pcb.yaml'), '--name', run_name,
    '--model', str(work / 'yolov8n.pt'), '--imgsz', '640',
    '--epochs', '100', '--batch', '8', '--device', '0', '--workers', '2'
], cwd=work, check=True)
weights = work / 'experiments' / run_name / 'weights' / 'best.pt'
if not weights.is_file(): raise FileNotFoundError(f'Training did not create {weights}')
print('Candidate checkpoint:', weights, 'bytes:', weights.stat().st_size)

In [ ]:
# Evaluate once on the official held-out test split after training is complete.
from ultralytics import YOLO
model = YOLO(str(weights))
result = model.val(data=str(processed / 'deep_pcb.yaml'), split='test', imgsz=640, batch=8, device='0', plots=True, project='/kaggle/working/pcb_evaluation', name=run_name, exist_ok=True)
per_class = {str(result.names[int(class_id)]): {'AP50': round(float(result.box.ap50[i]), 4), 'AP50_95': round(float(result.box.ap[i]), 4)} for i, class_id in enumerate(result.box.ap_class_index)}
metrics = {'exp_id': run_name, 'dataset': 'DeepPCB', 'split': 'official test', 'images': int(summary['splits']['test']['images']), 'imgsz': 640, 'P': round(float(result.box.mp), 4), 'R': round(float(result.box.mr), 4), 'mAP50': round(float(result.box.map50), 4), 'mAP50_95': round(float(result.box.map), 4), 'per_class': per_class, 'note': 'Kaggle GPU evaluation; measured only when this cell completes.'}
metrics_path = work / 'experiments' / run_name / 'pcb_test_metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

In [ ]:
# Download only the experiment artifacts and measured metrics.
archive = Path('/kaggle/working/PCB-EXP-001.zip')
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    exp = work / 'experiments' / run_name
    for item in exp.rglob('*'):
        if item.is_file(): zf.write(item, item.relative_to(work))
    zf.write(processed / 'summary.json', 'dataset_summary.json')
print('Download from Kaggle output:', archive, 'bytes:', archive.stat().st_size)